# Eksperimen Klasifikasi

Notebook ini membandingkan dua model, yaitu **Logistic Regression** dan **Random Forest**, pada beberapa jenis masalah klasifikasi. Dataset yang digunakan berbeda-beda, sehingga setiap eksperimen disesuaikan dengan bentuk target yang tersedia.

Tujuan utamanya bukan mencari satu model yang selalu paling bagus, tetapi melihat bagaimana kedua model bekerja pada jenis masalah dan data yang berbeda. Hasil akhirnya disimpan ke `hasil_klasifikasi.csv` agar lebih mudah diperiksa kembali.

## Gambaran alur

1. Memuat library dan menyiapkan fungsi preprocessing.
2. Membuat fungsi untuk menjalankan model sesuai jenis klasifikasinya.
3. Membagi data menjadi training dan testing.
4. Melatih model, membuat prediksi, lalu menghitung metrik evaluasi.
5. Menggabungkan hasil dari seluruh eksperimen.

**Catatan:** nilai metrik hanya menggambarkan performa pada data testing dalam skema pembagian yang digunakan. Hasil ini belum otomatis membuktikan bahwa model akan memiliki performa yang sama pada data baru di luar dataset.

## 1. Library yang digunakan

Library di bagian ini dipakai untuk mengolah data, membagi dataset, membangun pipeline preprocessing, melatih model, dan menghitung hasil evaluasi. Pipeline membantu memastikan preprocessing yang dipelajari dari data training diterapkan dengan cara yang sama pada data testing.

In [1]:
import re, warnings, collections
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, hamming_loss)

## 2. Pengaturan awal

Random state dibuat tetap agar pembagian data dan hasil eksperimen dapat diulang. Daftar `rows` menjadi tempat menyimpan metrik dari setiap kombinasi dataset dan model. Peringatan disembunyikan agar output lebih ringkas; jika sedang melakukan pemeriksaan atau debugging, peringatan sebaiknya ditampilkan kembali.

In [2]:
warnings.filterwarnings("ignore")
D = ""  # File CSV dibaca dari folder yang sama dengan notebook.
RS = 42
rows = []

## 3. Preprocessing fitur

Fitur numerik diisi menggunakan median, lalu distandardisasi. Median dipilih karena relatif tidak mudah terpengaruh nilai ekstrem. Fitur kategorikal diisi dengan nilai yang paling sering muncul dan diubah menjadi representasi numerik menggunakan one-hot encoding. `handle_unknown="ignore"` membantu pipeline tetap berjalan ketika data testing memiliki kategori yang tidak muncul saat training.

In [3]:
def prep(num, cat):
    t = []
    if num:
        t.append(("n", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num))
    if cat:
        t.append(("c", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                                 ("o", OneHotEncoder(handle_unknown="ignore"))]), cat))
    return ColumnTransformer(t)

## 4. Model yang dibandingkan

Dua model yang digunakan adalah Logistic Regression sebagai model linear dan Random Forest sebagai model berbasis kumpulan decision tree. Keduanya dipakai agar perbandingan tidak hanya bertumpu pada satu pendekatan. Pada kasus multilabel, Logistic Regression dibungkus dengan One-vs-Rest supaya model dapat memprediksi beberapa label sekaligus.

In [4]:
def models(kind):
    lr = LogisticRegression(max_iter=2000, class_weight="balanced")
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=RS)
    if kind == "multilabel":
        return {"Logistic Regression": OneVsRestClassifier(lr), "Random Forest": rf}
    return {"Logistic Regression": lr, "Random Forest": rf}

## 5. Binary Classification

Klasifikasi biner digunakan ketika target hanya memiliki dua kelas. Fungsi berikut melatih kedua model pada data training, kemudian mengukur hasil prediksi pada data testing. Untuk eksperimen biasa, pembagian data dilakukan secara stratified agar proporsi kelas target relatif terjaga. Jika data memiliki urutan waktu, pembagian dilakukan berdasarkan urutan tersebut agar data masa depan tidak masuk ke training.

In [5]:
# ---------------- BINER ----------------
def binary(name, X, y, num, cat, split=None):
    if split is None:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, stratify=y, random_state=RS)
    else:  # Untuk data berurutan berdasarkan waktu, data lama dipakai untuk training dan data setelahnya untuk testing.
        k = int(len(X) * .8); Xtr, Xte, ytr, yte = X[:k], X[k:], y[:k], y[k:]
    for m, clf in models("binary").items():
        p = Pipeline([("p", prep(num, cat)), ("m", clf)]).fit(Xtr, ytr)
        pr, pp = p.predict(Xte), p.predict_proba(Xte)[:, 1]
        rows.append(dict(Tipe="Biner", Dataset=name, Model=m, N=len(X), Positif=f"{y.mean():.1%}",
                         Accuracy=accuracy_score(yte, pr), Precision=precision_score(yte, pr, zero_division=0),
                         Recall=recall_score(yte, pr), F1=f1_score(yte, pr),
                         ROC_AUC=roc_auc_score(yte, pp), PR_AUC=average_precision_score(yte, pp)))

### Dataset untuk klasifikasi biner

Tiga eksperimen di bawah menggunakan target yang berbeda: status kebangkrutan perusahaan, arah pergerakan Bitcoin, dan kegagalan mesin. Beberapa kolom sengaja tidak dimasukkan sebagai fitur ketika berpotensi membocorkan informasi target. Langkah ini penting karena model yang menerima informasi yang seharusnya tidak tersedia saat prediksi dapat terlihat sangat akurat, tetapi hasilnya tidak realistis.

In [6]:
d = pd.read_csv(D + "data.csv"); d.columns = d.columns.str.strip()
y = d["Bankrupt?"]; X = d.drop(columns="Bankrupt?")
binary("Company Bankruptcy (data.csv)", X, y, list(X.columns), [])

b = pd.read_csv(D + "Bitcoin Market Analysis Dataset (2021-2025).csv", parse_dates=["date"]).sort_values("date")
b = b.dropna(subset=["target_direction"]).reset_index(drop=True)
y = b["target_direction"].astype(int)
X = b.drop(columns=["date", "target_direction", "target_return"])  # target_return tidak disertakan karena terlalu dekat dengan informasi target arah harga.
binary("Bitcoin Direction (split waktu)", X, y, list(X.columns), [], split="time")

a = pd.read_csv(D + "ai4i_predictive_maintenance.csv")
fail = ["TWF", "HDF", "PWF", "OSF", "RNF"]
y = a["Machine failure"]
X = a.drop(columns=["UDI", "Product ID", "Machine failure"] + fail)  # Kolom mode kegagalan tidak dipakai sebagai fitur karena berkaitan langsung dengan label kegagalan.
binary("AI4I Machine Failure", X, y, [c for c in X if c != "Type"], ["Type"])

## 6. Multiclass Classification

Pada klasifikasi multiclass, target memiliki lebih dari dua kelas dan setiap baris hanya masuk ke satu kelas. Data dibagi secara stratified agar kelas-kelas target tetap terwakili pada data training dan testing. Selain accuracy, F1 macro dan F1 weighted digunakan karena keduanya memberi sudut pandang berbeda ketika jumlah data tiap kelas tidak seimbang.

In [7]:
# ---------------- MULTICLASS ----------------
def multiclass(name, X, y, num, cat):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, stratify=y, random_state=RS)
    for m, clf in models("multiclass").items():
        p = Pipeline([("p", prep(num, cat)), ("m", clf)]).fit(Xtr, ytr)
        pr = p.predict(Xte)
        rows.append(dict(Tipe="Multiclass", Dataset=name, Model=m, N=len(X), Kelas=y.nunique(),
                         Accuracy=accuracy_score(yte, pr),
                         F1_macro=f1_score(yte, pr, average="macro"),
                         F1_weighted=f1_score(yte, pr, average="weighted"),
                         Precision_macro=precision_score(yte, pr, average="macro", zero_division=0),
                         Recall_macro=recall_score(yte, pr, average="macro")))

### Dataset gejala penyakit

Dataset ini menggunakan kolom gejala sebagai fitur dan `Disease` sebagai target. Dengan demikian, model mencoba memetakan kombinasi gejala ke satu kelas penyakit. Hasilnya perlu dibaca sebagai eksperimen pada dataset yang tersedia, bukan sebagai alat diagnosis.

In [8]:
x = pd.read_csv(D + "dataset1.csv")
multiclass("Disease dari gejala (dataset1)", x.drop(columns="Disease"), x["Disease"],
           [c for c in x if c != "Disease"], [])

### Mengubah nilai teks menjadi angka

Sebagian kolom pada dataset Kaggle tersimpan sebagai teks, termasuk ukuran file yang memiliki satuan. Fungsi berikut mengubah nilai yang dapat dibaca menjadi angka. Nilai yang formatnya tidak sesuai akan menjadi `NaN` dan selanjutnya ditangani oleh preprocessing.

In [9]:
def to_num(s, unit=False):
    s = str(s).strip()
    if unit:
        m = re.fullmatch(r"([\d.]+)\s*(B|kB|MB|GB|TB)", s)
        return float(m[1]) * {"B": 1, "kB": 1e3, "MB": 1e6, "GB": 1e9, "TB": 1e12}[m[2]] if m else np.nan
    try: return float(s)
    except ValueError: return np.nan

### Dataset kategori Kaggle

Eksperimen ini memprediksi kategori Gold, Silver, atau Bronze. Beberapa kolom teks diubah menjadi fitur numerik sederhana, misalnya jumlah jenis file dan penanda apakah format tertentu tercantum. `Total Votes` tidak dimasukkan sebagai fitur karena berpotensi menjadi sumber kebocoran target. Penyaringan kategori dilakukan agar eksperimen berfokus pada tiga kategori tersebut.

In [10]:
k = pd.read_csv(D + "Kaggle Dataset.csv", dtype=str)
k = k[k["Category"].isin(["Gold", "Silver", "Bronze"])].copy()  # Hanya kategori yang dipakai dalam eksperimen ini yang dipertahankan.
k["upd"] = k["Last Updated in number of days"].map(to_num)
k["usab"] = k["Usability Score"].map(to_num)
k["nfile"] = k["Number of File"].map(to_num)
k["size"] = k["Size"].map(lambda s: to_num(s, True))
k["votes"] = k["Total Votes"].map(to_num)
k["ntype"] = k["Type of file"].fillna("").str.count(",") + 1
for t in ["CSV", "JSON", "SQLite", "other"]:
    k["has_" + t] = k["Type of file"].fillna("").str.contains(t, case=False).astype(int)
feat_k = ["upd", "usab", "nfile", "size", "ntype"] + ["has_" + t for t in ["CSV", "JSON", "SQLite", "other"]]
# Total Votes tidak dipakai pada eksperimen ini karena kategori kemungkinan dibentuk dari jumlah votes. Memakainya sebagai fitur dapat membuat evaluasi terlalu optimistis.
multiclass("Kaggle Category (tanpa votes)", k[feat_k], k["Category"], feat_k, [])

### Dataset efisiensi energi

Nilai `Y1` dibagi menjadi tiga kelompok berdasarkan kuantil, yaitu Rendah, Sedang, dan Tinggi. Pembagian ini mengubah target numerik menjadi target multiclass. Fitur `Y1` dan `Y2` tidak dimasukkan ke dalam fitur prediksi; `Y1` adalah sumber pembentukan label, sedangkan `Y2` merupakan target lain yang berhubungan dengan beban energi.

In [11]:
e = pd.read_csv(D + "ENB2012_data.csv")
y = pd.qcut(e["Y1"], 3, labels=["Rendah", "Sedang", "Tinggi"])  # Nilai beban pemanas dibagi menjadi tiga kelompok berdasarkan kuantil: rendah, sedang, dan tinggi.
Xe = e.drop(columns=["Y1", "Y2"])  # Y2 tidak digunakan karena merupakan target lain yang berhubungan erat dengan Y1.
multiclass("Energy Efficiency: beban pemanas", Xe, y.astype(str), list(Xe.columns), [])

## 7. Multilabel Classification

Berbeda dari dua jenis klasifikasi sebelumnya, satu baris pada multilabel dapat memiliki lebih dari satu label sekaligus. Karena itu, target disimpan dalam bentuk matriks indikator 0 dan 1. Fungsi di bawah melatih model dan menghitung metrik yang sesuai untuk prediksi multilabel.

In [12]:
# ---------------- MULTILABEL ----------------
def multilabel(name, X, Y, num, cat):
    Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=.2, random_state=RS)
    for m, clf in models("multilabel").items():
        p = Pipeline([("p", prep(num, cat)), ("m", clf)]).fit(Xtr, Ytr)
        pr = p.predict(Xte)
        rows.append(dict(Tipe="Multilabel", Dataset=name, Model=m, N=len(X), Label=Y.shape[1],
                         Accuracy=accuracy_score(Yte, pr),  # Pada multilabel, accuracy adalah subset accuracy: semua label pada satu baris harus tepat agar dihitung benar.
                         F1_micro=f1_score(Yte, pr, average="micro", zero_division=0),
                         F1_macro=f1_score(Yte, pr, average="macro", zero_division=0),
                         Precision_micro=precision_score(Yte, pr, average="micro", zero_division=0),
                         Recall_micro=recall_score(Yte, pr, average="micro", zero_division=0),
                         Hamming_loss=hamming_loss(Yte, pr)))

### Dataset mode kegagalan mesin

Fitur identitas mesin dihapus karena tidak menjelaskan kondisi operasional yang ingin dipelajari. Kolom `Machine failure` dan lima kolom mode kegagalan dipisahkan dari fitur input. Eksperimen ini menggunakan lima mode kegagalan sebagai target multilabel, sehingga satu mesin dapat memiliki lebih dari satu label.

In [13]:
Xa = a.drop(columns=["UDI", "Product ID", "Machine failure"] + fail)
multilabel("AI4I Mode Kegagalan (5 label)", Xa, a[fail], [c for c in Xa if c != "Type"], ["Type"])

### Dataset gejala dan terapi

Kolom gejala diubah menjadi fitur biner: nilai 1 menandakan gejala tercantum dan 0 berarti tidak tercantum. Target terapi juga diubah menjadi beberapa label. Agar label yang terlalu jarang tidak mendominasi eksperimen, hanya terapi yang muncul setidaknya delapan kali yang dipertahankan. Baris tanpa label terapi yang tersisa dikeluarkan dari eksperimen.

In [14]:
s = pd.read_csv(D + "Diseases_Symptoms.csv").dropna()
norm = lambda v: sorted({t.strip().lower() for t in v.split(",") if t.strip()})
S = MultiLabelBinarizer(); Xs = pd.DataFrame(S.fit_transform(s["Symptoms"].map(norm)), columns=S.classes_)
cnt = collections.Counter(t for v in s["Treatments"].map(norm) for t in v)
top = {t for t, c in cnt.items() if c >= 8}
T = MultiLabelBinarizer(classes=sorted(top))
Ys = pd.DataFrame(T.fit_transform(s["Treatments"].map(lambda v: [t for t in norm(v) if t in top])), columns=T.classes_)
keep = Ys.sum(1) > 0
Xs, Ys = Xs[keep.values].reset_index(drop=True), Ys[keep.values].reset_index(drop=True)
Xs = Xs.loc[:, Xs.sum() >= 2]
multilabel("Diseases: gejala -> terapi", Xs, Ys, list(Xs.columns), [])

### Dataset jenis file Kaggle

Pada eksperimen ini, informasi jenis file diperlakukan sebagai target multilabel karena satu dataset dapat memiliki beberapa format sekaligus. Label yang jumlah kemunculannya kurang dari 30 tidak disertakan. Fitur `votes` digunakan di sini sebagai fitur prediksi jenis file, berbeda dari eksperimen multiclass sebelumnya yang memprediksi kategori Kaggle.

In [15]:
Yk = k[["Type of file"]].fillna("").iloc[:, 0].map(lambda v: [t.strip() for t in v.split(",") if t.strip()])
mk = MultiLabelBinarizer(); Yk = pd.DataFrame(mk.fit_transform(Yk), columns=mk.classes_, index=k.index)
Yk = Yk.loc[:, Yk.sum() >= 30]
fk = ["upd", "usab", "nfile", "size", "votes"]
ok = Yk.sum(1) > 0
multilabel("Kaggle: jenis file (tipe file)", k.loc[ok, fk], Yk[ok], fk, [])

## 8. Rekap hasil

Seluruh metrik dari fungsi eksperimen dikumpulkan menjadi satu tabel dan disimpan ke file CSV. Tabel dicetak terpisah berdasarkan jenis klasifikasi agar lebih mudah dibaca.

### Cara membaca metrik

- **Accuracy**: proporsi prediksi yang benar. Pada multilabel, nilai ini adalah *subset accuracy*, sehingga seluruh label pada satu baris harus cocok.
- **Precision**: dari prediksi positif yang dibuat model, seberapa banyak yang benar.
- **Recall**: dari seluruh kasus positif sebenarnya, seberapa banyak yang berhasil ditemukan model.
- **F1**: gabungan precision dan recall. F1 macro memberi bobot sama pada setiap kelas/label, sedangkan F1 weighted mempertimbangkan jumlah data pada tiap kelas.
- **ROC-AUC** dan **PR-AUC**: mengukur kemampuan pemisahan kelas berdasarkan skor prediksi, bukan hanya label hasil keputusan akhir.
- **Hamming loss**: proporsi keputusan label yang salah pada kasus multilabel; semakin kecil nilainya, semakin baik.

Saat membandingkan model, jangan hanya melihat accuracy. Perhatikan juga keseimbangan precision dan recall, distribusi kelas, serta jenis kesalahan yang paling penting untuk konteks dataset. Metrik antarjenis klasifikasi juga tidak selalu dapat dibandingkan secara langsung karena definisi dan bentuk targetnya berbeda.

In [16]:
# ---------------- HASIL ----------------
R = pd.DataFrame(rows)
R.to_csv("hasil_klasifikasi.csv", index=False)
pd.set_option("display.width", 250, "display.max_columns", 30)
for t in ["Biner", "Multiclass", "Multilabel"]:
    print("\n=====", t)
    print(R[R.Tipe == t].dropna(axis=1, how="all").drop(columns="Tipe").round(3).to_string(index=False))


===== Biner
                        Dataset               Model     N Positif  Accuracy  Precision  Recall    F1  ROC_AUC  PR_AUC
  Company Bankruptcy (data.csv) Logistic Regression  6819    3.2%     0.878      0.186   0.818 0.303    0.917   0.319
  Company Bankruptcy (data.csv)       Random Forest  6819    3.2%     0.971      0.692   0.205 0.316    0.941   0.444
Bitcoin Direction (split waktu) Logistic Regression  1826   52.0%     0.533      0.507   0.598 0.549    0.530   0.490
Bitcoin Direction (split waktu)       Random Forest  1826   52.0%     0.464      0.402   0.259 0.315    0.519   0.459
           AI4I Machine Failure Logistic Regression 10000    6.4%     0.774      0.199   0.836 0.322    0.875   0.459
           AI4I Machine Failure       Random Forest 10000    6.4%     0.967      0.970   0.500 0.660    0.941   0.756

===== Multiclass
                         Dataset               Model    N  Accuracy  Kelas  F1_macro  F1_weighted  Precision_macro  Recall_macro
  Disease dari

## 9. Catatan sebelum menarik kesimpulan

Notebook ini membandingkan model pada beberapa dataset dengan karakteristik dan target yang tidak sama. Karena itu, kesimpulan sebaiknya dibuat per eksperimen, bukan dengan menyatakan satu model paling unggul untuk semua kasus. Periksa juga distribusi kelas, jumlah data, potensi kebocoran target, dan keterbatasan dataset sebelum menggunakan hasil untuk keputusan nyata.